Import simulation outputs from LISFLOOD-FP and package as NetCDF files.

Ensure to specify `voutput` in LISFLOOD-FP parameters file to get velocity as well as water depth.

In [ ]:
import numpy as np
import os
import re
import rioxarray as rxr
import shutil
from typing import cast, Literal
import xarray as xr

import graph_creation

# LISFLOOD_OUTPUT_DIR = "/home/aidan/code/data/res_5m_training_data"
# DEM_FILE = "/home/aidan/code/mSWE-GNN/database/raw_datasets_dyce/DEM/DEM_0.xyz"
# POLYGON_FILE = "raw_datasets_dyce/Geometry/dyce_polygon.pol"
# PREFIX = "res_5m_acc_cuda"
# MAX_STEP = 400

# LISFLOOD_OUTPUT_DIR = "/home/aidan/code/data/res_dk15_hydrograph102"
# DEM_FILE = "/home/aidan/code/data/res_dk15_hydrograph102/res_dk15_hydrograph102.dem"
# POLYGON_FILE = "raw_datasets_dyce/Geometry/polygon_0.pol"
# PREFIX = "res_dk15_hydrograph102"
# MAX_STEP = 97

# NAME = "hydrograph_0_highflow"
LISFLOOD_OUTPUT_DIR = f"/home/aidan/code/tmp/"
DEM_FILE = f"/home/aidan/code/mSWE-GNN/database/raw_datasets_dyce/DEM/dyce_lisfloodfp.xyz"
POLYGON_FILE = "/home/aidan/code/mSWE-GNN/database/raw_datasets_dyce/Geometry/dyce_polygon.pol"
PREFIX = "result_hydrograph_0_highflow"
# MAX_STEP = 1081

LOAD_MESHES_FROM_FILE = True # Set to False to create meshes from polygon file (takes a long time, so set to True to load from file instead)

def read_step(step: int, prefix: str, ftype:Literal["wd", "wdfp", "elev", "Vx", "Vy"]|None=None) -> xr.DataArray:
        return cast(xr.DataArray, rxr.open_rasterio(os.path.join(LISFLOOD_OUTPUT_DIR, f"{prefix}-{int(step):04}.{ftype}"), parse_coordinates=True, masked=True))[0]

In [ ]:
def extract_parameter(ftype:Literal["wd", "wdfp", "elev", "Vx", "Vy"], max_step, prefix=PREFIX, shape=None):
    results = []
    for step in range(0, max_step+1):
        results.append(read_step(step, prefix, ftype))
    parameter_array = xr.concat(results, "time")
    
    return parameter_array

In [ ]:
if LOAD_MESHES_FROM_FILE:
    import pickle
    with open("dyce_mesh.pkl", "rb") as f:
        meshes = pickle.load(f)
else:
    meshes = graph_creation.create_mesh_dhydro(POLYGON_FILE, 8, False)
    # meshes = graph_creation.create_mesh_dhydro("raw_datasets_dk15/Geometry/polygon_102.pol", 1, False)

meshes = meshes[:6] # First 6 meshes gives 173056 faces
mesh = meshes[-1] # Highest resolution mesh

In [ ]:
meshes

In [ ]:
for mesh in meshes:
    mesh._import_DEM(DEM_FILE)

In [ ]:
boundary_edge_mask = mesh.edge_type == 1 # Mask out non-boundary edges (edges of type 1)

# Find point source node for input boundary conditions
pointsource_x, pointsource_y = 388911, 814101 # The true pointsource coordinates
# pointsource_x, pointsource_y = 389000, 813950 # Start of the river on the smooth-edged DEM
edge_xy = (mesh.node_xy[mesh.edge_index[1]] + mesh.node_xy[mesh.edge_index[0]]) / 2
dist_to_pointsource = np.ma.array(np.abs(edge_xy[:,0]-pointsource_x) + np.abs(edge_xy[:,1]-pointsource_y), mask=boundary_edge_mask)
bc_edge_index = np.argmin(dist_to_pointsource)
print(edge_xy[bc_edge_index])
print("Distance: ", np.sqrt(np.sum((edge_xy[bc_edge_index] - np.array([pointsource_x, pointsource_y]))**2)) )

# Set edge type to BC edge
mesh.edge_type[bc_edge_index] = 2

# Correctly format edge_faces
edge_faces = mesh.edge_faces.reshape(-1,2)
if edge_faces[bc_edge_index][0] != -1:
    # If the first -1 (indicating no face on this side) isn't first
    edge_faces[bc_edge_index] = edge_faces[bc_edge_index, ::-1] # Swap the node indices of the edges (graph_creation.Mesh._import_from_map_netcdf relies on this to identify the BC edge)


In [ ]:
import matplotlib.pyplot as plt
%matplotlib qt
fig, ax = plt.subplots()
meshes[-1].meshtmp.plot_faces(ax)
ax.scatter(pointsource_x, pointsource_y, color="red")
plt.show()

In [ ]:
face_coords = xr.Dataset(
    coords={
        "mesh2d_nFaces": ("mesh2d_nFaces", range(mesh.face_xy.shape[0])),
        "x": ("mesh2d_nFaces", mesh.face_xy[:, 0]),
        "y": ("mesh2d_nFaces", mesh.face_xy[:, 1]),
    }
)

def extract_parameters(max_step, prefix):
    print("Extracting water depths")
    wd = extract_parameter("wd", max_step, prefix=prefix)
    print("Extracting x velocities")
    Vx = extract_parameter("Vx", max_step, shape=wd.shape[1:], prefix=prefix)
    print("Extracting y velocities")
    Vy = extract_parameter("Vy", max_step, shape=wd.shape[1:], prefix=prefix)
    return wd, Vx, Vy

In [ ]:
def generate_dataset(wd, Vx, Vy, mesh, hydrograph_name):
    # The +1 on variables which are indicies is expected by graph_creation on loading in the NetCDF file.

    simulation_output = xr.Dataset({
        "mesh2d_node_x": xr.DataArray(mesh.node_x, dims=["mesh2d_nNodes"]),
        "mesh2d_node_y": xr.DataArray(mesh.node_y, dims=["mesh2d_nNodes"]),
        "mesh2d_face_x": xr.DataArray(mesh.face_x, dims=["mesh2d_nFaces"]),
        "mesh2d_face_y": xr.DataArray(mesh.face_y, dims=["mesh2d_nFaces"]),
        "mesh2d_edge_nodes": xr.DataArray(mesh.edge_index.T + 1, dims=["mesh2d_nEdges", "Two"]),
        
        "mesh2d_edge_type": xr.DataArray(mesh.edge_type, dims=["mesh2d_nEdges"]),
        "mesh2d_edge_faces": xr.DataArray(edge_faces + 1, dims=["mesh2d_nEdges", "Two"]),
        "mesh2d_face_nodes": xr.DataArray(graph_creation.get_face_nodes_mesh(mesh) + 1, dims=["mesh2d_face_nodes", "mesh2d_nMax_face_nodes"]),

        "mesh2d_waterdepth": wd.sel(x=face_coords["x"], y=face_coords["y"], method="nearest").reset_coords(drop=True),
        "mesh2d_ucx": Vx.sel(x=face_coords["x"], y=face_coords["y"], method="nearest").reset_coords(drop=True),
        "mesh2d_ucy": Vy.sel(x=face_coords["x"], y=face_coords["y"], method="nearest").reset_coords(drop=True),

        "mesh2d_dem": xr.DataArray(mesh.DEM, dims=["mesh2d_nFaces"]),
    })

    simulation_output.reset_index("mesh2d_nFaces").to_netcdf(f"{hydrograph_name}.nc", format="NETCDF4")

In [ ]:
SOURCE_FOLDER = "/one/exageo/data/dyce-lisflood-fp-cpu-training-data/"
DEST_FOLDER = "/home/aidan/code/mSWE-GNN/database/raw_datasets_dyce/Simulations_6meshes"
TMP_FOLDER = "/home/aidan/code/tmp/"

In [ ]:
completed_hydrographs = set([file.rstrip(".nc.zst") for file in os.listdir(DEST_FOLDER) if file.startswith("hydrograph") and file.endswith(".nc.zst")])

In [ ]:
todo = set([file.rstrip(".tar.zst") for file in os.listdir(SOURCE_FOLDER) if file.endswith(".tar.zst")]) - completed_hydrographs

while todo:
    hydrograph_name = min(todo)
    completed_hydrographs.add(hydrograph_name)
    todo = set([file.rstrip(".tar.zst") for file in os.listdir(SOURCE_FOLDER) if file.endswith(".tar.zst")]) - completed_hydrographs
    with open("broken_hydrographs.txt") as f:
        if hydrograph_name in f.read():
            print(f"Skipping hydrograph {hydrograph_name} as it is marked as broken in broken_hydrographs.txt")
            continue
    
    print(f"Processing hydrograph {hydrograph_name}")
    print(f"Copying and extracting {hydrograph_name}.tar.zst to {TMP_FOLDER}")
    os.system(f"tar --zstd -xvf {os.path.join(SOURCE_FOLDER, f'{hydrograph_name}.tar.zst')} -C {TMP_FOLDER} >/dev/null")
    
    max_step = re.findall(r'.*-(\d\d\d\d)\.wd', "\n".join(os.listdir(os.path.join(TMP_FOLDER,f"result_{hydrograph_name}"))))
    max_step = int(sorted(max_step)[-1])
    max_step_expected = int(int(os.popen(f"tail -n 1 /home/aidan/code/flooddata-gen/data/dyce_hydrographs/{hydrograph_name}.bdy | awk '{{print $2}}'").read()) / 180)
    print(f"Expected final timestep: {max_step_expected}. Actual final timestep: {max_step}")
    if max_step != max_step_expected:
        print(f"WARNING: Expected final timestep {max_step_expected} does not match actual final timestep {max_step} for hydrograph {hydrograph_name}. Skipping this hydrograph.")
        os.system(f"echo WARNING: Expected final timestep {max_step_expected} does not match actual final timestep {max_step} for hydrograph {hydrograph_name}. Skipping this hydrograph. >> broken_hydrographs.txt")
        print(f"Cleaning up extracted folder")
        os.system(f"rm -r {os.path.join(TMP_FOLDER, f'result_{hydrograph_name}')}")
        print("Done\n")
        continue
    
    print(f"Extracting parameters for {hydrograph_name} ({max_step+1} timesteps)")
    try:
        wd, Vx, Vy = extract_parameters(max_step, prefix=f"result_{hydrograph_name}/result_{hydrograph_name}")
    except Exception as e:
        print(f"WARNING: hydrograph {hydrograph_name} gave error {e}. Skipping this hydrograph.")
        os.system(f"echo WWARNING: hydrograph {hydrograph_name} gave error {e}. Skipping this hydrograph. >> broken_hydrographs.txt")
        print(f"Cleaning up extracted folder")
        os.system(f"rm -r {os.path.join(TMP_FOLDER, f'result_{hydrograph_name}')}")
        print("Done\n")
        continue

    print(f"Generating dataset for {hydrograph_name}")
    generate_dataset(wd, Vx, Vy, mesh, hydrograph_name)
    
    print(f"Compressing dataset {hydrograph_name}.nc")
    os.system(f"zstd -10 {hydrograph_name}.nc")
    os.rename(f"{hydrograph_name}.nc.zst", os.path.join(DEST_FOLDER, f"{hydrograph_name}.nc.zst"))
    print(f"Cleaning up extracted folder and uncompressed .nc file")
    os.system(f"rm -r {os.path.join(TMP_FOLDER, f'result_{hydrograph_name}')}")
    os.system(f"rm {hydrograph_name}.nc")
    print("Done.\n")


## Zarr Output
Code to output to zarr rather than compressed NetCDF4 files:

In [ ]:
import xarray as xr
import numcodecs

ds = <Xarray Dataset generated in generate_dataset()>

flowvars_encoding = {"compressor":numcodecs.Zstd(level=19)(level=19), "scale_factor":0.001, "dtype":"int16", "filters":[numcodecs.Delta(dtype="int16")]}
coordinate_encoding = {"compressor":numcodecs.Zstd(level=19)(level=19), "dtype": "float64"}
elevation_encoding = coordinate_encoding
index_encoding = {"compressor":numcodecs.Zstd(level=19)(level=19), "dtype": "int32"}
smallint_encoding = {"compressor":numcodecs.Zstd(level=19)(level=19), "dtype": "int8"}

ds_chunked = ds.chunk({"time":64}) # Time chunks of 64 yields chunksizes of ~1MB. Aiming for the largest chunksize where reads (from a network drive) will still be seek-limited (the application favours small chunk sizes for random access)

ds_chunked.to_zarr(
    "hydrograph_0002_highflow.zarr",
    encoding={
        "mesh2d_node_x": coordinate_encoding,
        "mesh2d_node_y": coordinate_encoding,
        "mesh2d_face_x": coordinate_encoding,
        "mesh2d_face_y": coordinate_encoding,
        "mesh2d_edge_nodes": index_encoding,
        "mesh2d_edge_type": smallint_encoding,
        "mesh2d_edge_faces": index_encoding,
        "mesh2d_waterdepth": flowvars_encoding,
        "mesh2d_ucx": flowvars_encoding,
        "mesh2d_ucy": flowvars_encoding,
        "mesh2d_dem": elevation_encoding
    }
)